# CI/CD 第1周：概念与基础 — 跑通第一个 Pipeline

> **学习目标**：理解 CI/CD 理念，掌握 GitHub Actions 核心概念，能写一个从 push 到 test 的 workflow

---

## 开篇：为什么需要 CI/CD？

想象这个场景：你是一个三人小团队的后端开发者。周一早上，你提交了代码，然后同事也提交了代码。周三要上线了，你们开始合并代码——

**结果**：
- 你的改动和同事的改冲突了 5 个文件
- 合并后单元测试挂了 3 个
- 手动跑一遍测试花了 20 分钟
- 修复了 A 问题又冒出了 B 问题
- 最后上线推迟了两天

这就是典型的 **"集成地狱"**(Integration Hell)。CI/CD 就是用来解决这个问题的。

---

### 持续集成 (Continuous Integration / CI)

**核心理念**：频繁地将代码合并到主干（每天多次），每次合并都**自动构建 + 自动测试**。

```
传统方式：
写一周代码 → 合并 → 手动构建 → 手动测试 → 发现问题 → 花几天修复 → 再测 → 上线
                            ↑
                      集成地狱在这里！

CI 方式：
提交代码 → 自动构建 → 自动测试 → 通过 → 安心
                                     → 失败 → 立刻修 → 再提交
            整个过程几分钟完成，问题在刚出现时就发现
```

**CI 的核心收益**：
- 尽早发现集成问题（几分钟而不是几天）
- 每次提交都经过自动化检查，质量有保障
- 小步提交、频繁合并，减少冲突规模

### 持续交付 (Continuous Delivery / CD)

CI 通过后，**自动将代码部署到类生产环境（staging）**，但上线到生产需要人工审批。

```
CI 成功 → 自动部署到 staging → 人工审批 → 部署到生产
```

### 持续部署 (Continuous Deployment)

更进一步：每次 CI 通过后**自动上线到生产**，没有人工干预。

```
CI 成功 → 自动部署到生产
```

只有**自动化测试足够完善**（覆盖率高、可靠性强）的团队才适合持续部署。

#### 传统发布 vs CI/CD 对比

| 维度 | 传统方式 | CI/CD 方式 |
|------|----------|------------|
| 构建 | 手动在本地构建 | 自动在 CI 服务器构建 |
| 测试 | 手动跑测试 | 自动跑全套测试 |
| 部署 | 手动上传到服务器 | 自动化流水线部署 |
| 发现问题 | 上线后才发现 | 提交时就发现 |
| 回滚 | 手动恢复 | 一键回滚或自动回滚 |
| 频率 | 每周/每月一次 | 每天多次 |

### 练习：阅读真实项目的 GitHub Actions 配置

在浏览器中打开 [fastapi/fastapi](https://github.com/fastapi/fastapi) 仓库，查看 `.github/workflows/` 目录下的 YAML 文件。
先感受一下真实项目的 CI/CD 长什么样——不需要理解每一行，只需要有个印象。

---

## Day 2：GitHub Actions 核心概念

GitHub Actions 是 GitHub 自带的 CI/CD 平台。你在仓库里放一个 `.yml` 配置文件，GitHub 就会按照你的配置自动执行任务。

### 五大核心概念

```
┌────────────────────────────────────────────────────┐
│                    Workflow                         │
│  ┌──────────────────────────────────────────────┐  │
│  │  Job 1 (lint: ubuntu-latest)                 │  │
│  │  ┌─────────────┐ ┌─────────────┐             │  │
│  │  │ Step 1      │ │ Step 2       │            │  │
│  │  │ checkout    │ │ pip install  │            │  │
│  │  └─────────────┘ └─────────────┘             │  │
│  └──────────────────────────────────────────────┘  │
│  ┌──────────────────────────────────────────────┐  │
│  │  Job 2 (test: ubuntu-latest)                 │  │
│  │  ┌─────────────┐ ┌─────────────┐             │  │
│  │  │ Step 1      │ │ Step 2       │            │  │
│  │  │ checkout    │ │ pytest       │            │  │
│  │  └─────────────┘ └─────────────┘             │  │
│  └──────────────────────────────────────────────┘  │
└────────────────────────────────────────────────────┘
```

| 概念 | 说明 |
|------|------|
| **Workflow** | 一个完整的自动化流程，定义在 `.github/workflows/*.yml` |
| **Event / Trigger** | 什么触发了 workflow：`push`、`pull_request`、`schedule`、`workflow_dispatch` |
| **Job** | 一组在同一个 Runner 上执行的 Step，默认并行运行 |
| **Step** | 最小的执行单元：`run`（shell 命令）或 `uses`（复用 action） |
| **Runner** | 执行 Job 的机器：GitHub 托管（ubuntu/macos/windows）或自建 |

### Event 触发方式

| 触发方式 | YAML 写法 | 使用场景 |
|----------|-----------|----------|
| 代码推送 | `on: push` | 最常用，每次提交都触发 |
| PR 事件 | `on: pull_request` | 代码审查时自动检查 |
| 定时 | `on: schedule: - cron: "0 2 * * *"` | 每天凌晨跑测试 |
| 手动 | `on: workflow_dispatch` | 手动点击触发 |
| Tag 推送 | `on: push: tags: ["v*"]` | 发版时触发部署 |

### 练习

在浏览器打开你的 GitHub 仓库 → Actions 标签页，浏览 GitHub 推荐的工作流模板。
GitHub 会根据你仓库的语言推荐对应模板（Python、Node.js、Docker 等）。

---

## Day 3：第一个 Workflow

### YAML 语法速览

GitHub Actions 使用 YAML 格式。如果你不熟悉 YAML，先记住这几点：

| 要点 | 说明 |
|------|------|
| 缩进 | 2 空格，不能混用 Tab |
| 键值 | `key: value`（冒号后要有空格） |
| 列表 | `- item1` / `- item2`（横线后要有空格） |
| 多行字符串 | `\|`（保留换行）或 `>`（折叠成一行） |
| 表达式 | `${{ <expression> }}` |

### 最小的 Workflow

```yaml
# .github/workflows/ci.yml
name: CI
on: push
jobs:
  hello:
    runs-on: ubuntu-latest
    steps:
      - run: echo "Hello CI/CD!"
```

这个 workflow 会在每次 `git push` 时触发，在 ubuntu 机器上执行一条命令。

让我们在本地演示这个流程——虽然不能真正 push 到 GitHub，但我们可以在 Jupyter 中模拟 Workflow 的逻辑。

In [ ]:
# 模拟 workflow 执行流程
# 在真实场景中，这些步骤由 GitHub Runner 自动执行

import subprocess
import sys

print("=" * 50)
print("Workflow: CI")
print("Event: push")
print("Job: hello")
print("Runner: ubuntu-latest")
print("=" * 50)

# 模拟 Step 1
print("\n[Step 1/1] run: echo Hello CI/CD!")
result = subprocess.run(["echo", "Hello CI/CD!"], capture_output=True, text=True)
print(result.stdout.strip())

print("\n" + "=" * 50)
print("✅ Workflow 执行完成")
print("=" * 50)

In [ ]:
# 查看当前目录结构（模拟 checkout 的效果）
# checkout action 会把仓库代码 clone 到 Runner 上
import os
print("当前工作目录:", os.getcwd())
print("\n目录内容:")
for root, dirs, files in os.walk("."):
    level = root.replace(".", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")
    if level > 2:
        break

### 练习：创建你的第一个 Workflow

> 注意：以下步骤需要在你的本地 Git 仓库中操作，不能在 Jupyter 中完全模拟。

1. 在你的 GitHub 仓库中创建 `.github/workflows/ci.yml` 文件
2. 写入上面那个最小 workflow
3. `git add . && git commit -m "add ci workflow" && git push`
4. 在 GitHub 仓库的 Actions 标签页观察运行结果

这个练习的目标是让你体验一下从"写配置"到"看到运行"的完整流程。

---

## Day 4：Checkout 与多 Step 编排

### actions/checkout：把代码拉到 Runner

大多数 workflow 的第一步都是 checkout——没有代码，后面的测试、构建都无从谈起。

```yaml
steps:
  - name: Checkout code
    uses: actions/checkout@v4
```

`actions/checkout@v4` 是 GitHub 官方的 Action，功能包括：
- Clone 仓库代码到 Runner
- 默认 checkout 触发事件的 commit
- 支持 `fetch-depth`（只拉最近 1 个 commit 可以加速）
- 支持 `ref` 参数指定分支

### Step 的执行顺序和条件

同 Job 内的 Step 是**串行执行**的——前一步成功才能执行后一步。

但你可以用 `if` 控制条件：

| 条件函数 | 说明 |
|----------|------|
| `success()` | 上一步成功时执行（默认） |
| `failure()` | 上一步失败时执行 |
| `always()` | 不管成功失败都执行 |
| `cancelled()` | 工作流被取消时执行 |

```yaml
steps:
  - run: ./run_tests.sh
  - run: echo "测试通过！"
    if: success()
  - run: echo "测试失败了！"
    if: failure()
  - run: echo "不管结果如何都执行"
    if: always()
```

In [ ]:
# 在本地模拟 Step 的串行执行和条件控制

steps = [
    ("checkout", True),
    ("列出文件", True),
    ("python --version", True),
    ("打印当前时间", True),
]

print("执行 Workflow: checkout → 文件列表 → python版本 → 时间")
print("-" * 50)

for step_name, should_pass in steps:
    print(f"\n▶ 正在执行: {step_name}")
    import subprocess

    if step_name == "checkout":
        print("  ✅ actions/checkout@v4 (模拟)")
        print("  已获取仓库代码")
    elif step_name == "列出文件":
        result = subprocess.run(["ls", "-la"], capture_output=True, text=True)
        print(f"  {result.stdout}")
    elif step_name == "python --version":
        result = subprocess.run([sys.executable, "--version"], capture_output=True, text=True)
        print(f"  {result.stdout.strip()}")
    elif step_name == "打印当前时间":
        from datetime import datetime
        print(f"  当前时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print("\n" + "✅ 所有步骤执行完成")

### GITHUB_WORKSPACE 环境变量

GitHub Actions 会自动设置一些环境变量，在你的 Step 中可以直接使用：

| 环境变量 | 说明 |
|----------|------|
| `GITHUB_WORKSPACE` | 代码在 Runner 上的存放路径 |
| `GITHUB_SHA` | 触发 workflow 的 commit SHA |
| `GITHUB_REF` | 触发 workflow 的分支或 tag 引用 |
| `GITHUB_REPOSITORY` | 仓库名（格式：owner/repo） |
| `RUNNER_OS` | Runner 操作系统 |

```yaml
steps:
  - run: |
      echo "仓库: ${{ github.repository }}"
      echo "分支: ${{ github.ref_name }}"
      echo "Commit: ${{ github.sha }}"
      echo "Runner: ${{ runner.os }}"
```

---

## Day 5：Runner 环境与 setup 类 Action

### GitHub 托管的 Runner

GitHub 提供三种操作系统类型的 Runner：

| Runner | 规格 | 适用场景 |
|--------|------|----------|
| `ubuntu-latest` | 2 核 CPU / 7GB RAM / 14GB SSD | 通用 |
| `macos-latest` | 3 核 CPU / 14GB RAM / 14GB SSD | iOS/macOS 构建 |
| `windows-latest` | 2 核 CPU / 7GB RAM / 14GB SSD | .NET / Windows 专用 |

**Ubuntu Runner 预装软件**（部分）：
- Python 3.x
- Node.js
- Docker
- Git
- curl / wget
- Make / CMake
- OpenJDK
- SQLite / PostgreSQL / MySQL 客户端

### setup 类 Action

这些 Action 用于在 Runner 上安装和配置特定语言环境：

```yaml
# 设置 Python
- uses: actions/setup-python@v5
  with:
    python-version: "3.12"

# 设置 Node.js
- uses: actions/setup-node@v4
  with:
    node-version: "20"

# 设置 Java
- uses: actions/setup-java@v4
  with:
    distribution: "temurin"
    java-version: "17"
```

In [ ]:
# 模拟 setup-python 的效果
# 在本地查看 Python 信息

import sys
import platform

print("=" * 50)
print("模拟 actions/setup-python@v5")
print(f"Python 版本: {sys.version}")
print(f"Python 路径: {sys.executable}")
print(f"操作系统: {platform.system()} {platform.release()}")
print(f"架构: {platform.machine()}")
print("=" * 50)

In [ ]:
# 模拟 Runner 信息
import platform
import os

print("🏃 Runner 环境信息")
print("-" * 30)
print(f"操作系统: {platform.system()} {platform.release()}")
print(f"机器架构: {platform.machine()}")
print(f"处理器: {platform.processor()}")
print(f"主机名: {platform.node()}")
print(f"当前用户: {os.environ.get('USER', 'unknown')}")
print(f"当前路径: {os.getcwd()}")
print(f"Shell: {os.environ.get('SHELL', 'unknown')}")

# 检查预装软件
print("\n📦 预装软件检查:")
for cmd in ["python3 --version", "git --version", "docker --version", "curl --version"]:
    import subprocess
    result = subprocess.run(cmd.split(), capture_output=True, text=True)
    output = result.stdout.strip() or result.stderr.strip()
    print(f"  {output.split(chr(10))[0]}")

### 练习：带 setup-python 的 Workflow

写一个 workflow，完成以下步骤：
1. Checkout 代码
2. 用 `setup-python` 指定 Python 3.12
3. 运行一个 Python 脚本（打印 Python 版本、当前时间、仓库信息）

```yaml
name: Python Demo
on: push
jobs:
  demo:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - name: Run Python script
        run: python -c "
import sys, datetime
print(f'Python {sys.version}')
print(f'Today is {datetime.date.today()}')
"
```

---

## Day 6：多 Job 与依赖

### Job 的并行与串行

默认情况下，Workflow 中的所有 Job **并行执行**。但你可以用 `needs` 关键字控制依赖关系：

```yaml
jobs:
  job-a:
    steps:
      - run: echo "Job A"

  job-b:
    needs: job-a    # job-b 等 job-a 完成后才执行
    steps:
      - run: echo "Job B 依赖于 Job A"

  job-c:
    needs: job-a    # job-c 也等 job-a
    steps:
      - run: echo "Job C 也依赖于 Job A"
```

执行顺序：
```
     ┌──▶ job-b
job-a┤
     └──▶ job-c
```

job-a 先执行，成功后 job-b 和 job-c 并行执行。

### 练习：多 Job 编排

创建一个含 3 个 Job 的 Workflow，模拟实际项目的代码检查流程：

```
lint（代码格式检查）
  ├──▶ test（运行 pytest）
  └──▶ type-check（运行 mypy）
```

要求：
- `lint` 先跑，检查代码格式
- `lint` 成功后，`test` 和 `type-check` 并行跑
- 各 Job 用 `actions/checkout@v4` 拉取代码
- 用 `actions/setup-python@v5` 设置环境

让我们在本地模拟这个流程。

In [ ]:
# 在本地模拟多 Job 执行流程
import subprocess
import time

print("=" * 60)
print("🚀 Workflow 开始执行")
print("Event: push")
print("=" * 60)

# Job 1: lint
print("\n" + "=" * 40)
print("📋 Job 1/3: lint")
print("=" * 40)
time.sleep(0.5)
print("▶ Step 1: actions/checkout@v4 ✅")
print("▶ Step 2: actions/setup-python@v5 ✅")
print("▶ Step 3: ruff check .")
result = subprocess.run(["echo", "模拟 ruff 检查通过 ✅"], capture_output=True, text=True)
print(f"  {result.stdout.strip()}")
print("\n✅ Job lint 执行成功")

# Job 2: test (依赖 lint)
print("\n" + "=" * 40)
print("🧪 Job 2/3: test (依赖 lint)")
print("=" * 40)
time.sleep(0.3)
print("▶ Step 1: actions/checkout@v4 ✅")
print("▶ Step 2: pytest")
time.sleep(0.5)
result = subprocess.run(["echo", "模拟: 8 passed in 2.3s"], capture_output=True, text=True)
print(f"  {result.stdout.strip()}")
print("\n✅ Job test 执行成功")

# Job 3: type-check (依赖 lint)
print("\n" + "=" * 40)
print("🔍 Job 3/3: type-check (依赖 lint)")
print("=" * 40)
time.sleep(0.3)
print("▶ Step 1: actions/checkout@v4 ✅")
print("▶ Step 2: mypy .")
time.sleep(0.5)
result = subprocess.run(["echo", "模拟: 0 errors found ✅"], capture_output=True, text=True)
print(f"  {result.stdout.strip()}")
print("\n✅ Job type-check 执行成功")

print("\n" + "=" * 60)
print("🎉 所有 Job 执行完成！")
print("=" * 60)

### 跨 Job 传递数据

不同 Job 在不同的 Runner 上执行，默认不能共享数据。如果想传递数据，需要用到 **Artifacts**（下一周会详细讲）。

```yaml
jobs:
  build:
    steps:
      - run: echo "data" > output.txt
      - uses: actions/upload-artifact@v4
        with:
          name: my-output
          path: output.txt

  consume:
    needs: build
    steps:
      - uses: actions/download-artifact@v4
        with:
          name: my-output
      - run: cat output.txt
```

---

## 🎯 第1周总结

### 核心概念回顾

| 概念 | 一句话 |
|------|--------|
| **CI** | 频繁合并、自动构建测试，尽早发现问题 |
| **CD** | CI 通过后自动部署到目标环境 |
| **Workflow** | 定义在 `.github/workflows/*.yml` 的自动化流程 |
| **Event** | 触发 workflow 的事件（push、PR、schedule 等） |
| **Job** | 一组在相同 Runner 上串行执行的 Step |
| **Step** | 最小的执行单元（run 命令或 uses action） |
| **Runner** | 执行 Job 的机器 |
| **Needs** | 控制 Job 的执行顺序和依赖关系 |

### 常用命令/配置速查

```yaml
# 最简 workflow
name: CI
on: push
jobs:
  job-name:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: echo "Hello!"
```

```yaml
# 多 Job 依赖
jobs:
  lint:
    ...
  test:
    needs: lint
    ...
```

```yaml
# 条件执行
steps:
  - run: ./test.sh
  - run: echo "成功"
    if: success()
  - run: echo "失败"
    if: failure()
```

---

## 🧪 综合练习：完整的 Push 触发 CI 流程

为目标仓库创建完整的 **push 触发 → 环境准备 → 代码检查 → 单元测试** 流程。

### 练习要求

写一个 Workflow（`.github/workflows/full-ci.yml`），包含以下结构：

```
push 触发
  ├── Job: lint
  │     ├── checkout
  │     ├── setup-python
  │     └── ruff check .
  ├── Job: test (依赖 lint)
  │     ├── checkout
  │     ├── setup-python (Python 3.12)
  │     ├── pip install -r requirements.txt
  │     └── pytest
  └── Job: format-check (依赖 lint)
        ├── checkout
        ├── setup-python
        └── ruff format --check .
```

### 提示

1. 先在仓库中创建 `requirements.txt`，包含 `pytest`、`ruff` 等依赖
2. `ruff check .` 检查代码问题
3. `ruff format --check .` 检查格式（不自动修改）
4. `pytest` 运行单元测试
5. 如果任何 Job 失败，Push 会显示红色 ✗

### 在本地模拟

下面的代码模拟了这个综合练习的完整流程。

In [ ]:
# 模拟综合练习的完整 Workflow 执行

import time

workflow_yaml = '''
name: Full CI Pipeline
on: push

jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - run: pip install ruff
      - run: ruff check .

  test:
    needs: lint
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - run: pip install -r requirements.txt
      - run: pytest

  format-check:
    needs: lint
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - run: pip install ruff
      - run: ruff format --check .
'''

print("📄 Workflow 配置预览：")
print(workflow_yaml)
print("\n" + "=" * 60)
print("🚀 执行完整 CI Pipeline")
print("=" * 60)

# 模拟运行
jobs = [
    ("lint", "ruff check .", False, 0.8),
    ("test", "pytest", "lint", 1.2),
    ("format-check", "ruff format --check .", "lint", 0.6),
]

completed = set()
def run_job(name, cmd, dependency, duration):
    if dependency and dependency not in completed:
        print(f"\n⏳ Job '{name}' 等待 '{dependency}' 完成...")
        time.sleep(0.3)

    print(f"\n{'='*40}")
    print(f"▶ Job: {name}")
    print(f"{'='*40}")
    print(f"  步骤:")
    print(f"    1. actions/checkout@v4")
    print(f"    2. actions/setup-python@v5 (3.12)")
    print(f"    3. {cmd}")

    time.sleep(duration)
    print(f"\n  ✅ Job '{name}' 执行成功！")
    completed.add(name)

for name, cmd, dep, dur in jobs:
    run_job(name, cmd, dep, dur)

print("\n" + "=" * 60)
print("🎉 完整 CI Pipeline 全部通过！")
print("=" * 60)

### 提交到 GitHub 后的效果

当你把这个 workflow push 到 GitHub 后：

1. 在仓库页面点击 **Actions** 标签
2. 你会看到名为 "Full CI Pipeline" 的 workflow 正在运行
3. 点进去可以查看每个 Job 的实时日志
4. 全部通过后，commit 旁边会出现绿色 ✅
5. 如果有 Job 失败，会出现红色 ❌，并会发邮件通知你

这就是 CI 的基本工作流——**提交代码 → 自动检查 → 快速反馈**。